In [2]:
# Загрузка данных
import os
import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

connection = {"sslmode": "require", "target_session_attrs": "read-write"}
postgres_credentials = {
    "host": os.getenv("DB_DESTINATION_HOST"), 
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD"),
}
assert all([var_value != "" for var_value in list(postgres_credentials.values())])
connection.update(postgres_credentials)

TABLE_NAME = "real_estate_clean"

with psycopg.connect(**connection) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")
        data = cur.fetchall()
        columns = [col[0] for col in cur.description]

df = pd.DataFrame(data, columns=columns)
df.head(3)      

,id,flat_id,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price
0,1,0,1965,6,55.717113,37.781120,2.64,84,12,1,9,9.9,19.9,1,0,0,35.099998,9500000
1,2,1,2001,2,55.794849,37.608013,3.00,97,10,1,7,0.0,16.6,1,0,0,43.000000,13500000
2,3,2,2000,4,55.740040,37.761742,2.70,80,10,1,9,9.0,32.0,2,0,0,56.000000,13500000


In [5]:
# загрузка модели и метрик

import boto3
import joblib
import io
import json
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    endpoint_url=os.getenv('MLFLOW_S3_ENDPOINT_URL')
)
bucket = os.getenv('AWS_BUCKET_NAME')

# Загрузить модель
response = s3.get_object(Bucket=bucket, Key='models/fitted_model.pkl')
model_bytes = io.BytesIO(response['Body'].read())
model = joblib.load(model_bytes)
print(f"Модель загружена")
print(f"Тип: {type(model)}")
print(f"Pipeline steps: {list(model.named_steps.keys())}")
print()

try:
    check_is_fitted(model)
    print("Системная проверка пройдена: Pipeline находится в обученном состоянии.")
except NotFittedError:
    print("ВНИМАНИЕ: Pipeline НЕ обучен (потеряны веса или произошел сброс состояния).")

print()

# Загрузить метрики
response = s3.get_object(Bucket=bucket, Key='cv_results/cv_res.json')
metrics = json.loads(response['Body'].read())
print(f"Метрики загружены:")
for key, val in metrics.items():
    print(f"  {key}: {val}")

Модель загружена
Тип: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: ['preprocessor', 'model']

Системная проверка пройдена: Pipeline находится в обученном состоянии.

Метрики загружены:
  fit_time: 27.291
  score_time: 0.281
  test_r2: 0.868
  test_neg_mean_absolute_error: -3669801.22
  test_neg_root_mean_squared_error: -11654096.221


In [6]:
# логирование модели и метрик
import mlflow
from mlflow.models import infer_signature

EXPERIMENT_NAME = "sprint2_project"
RUN_NAME = "1_base_model"
REGISTRY_MODEL_NAME = "real_estate_model"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

# Отделяем целевую переменную, чтобы получить признаки, которые ожидает модель
X = df.drop(columns=['price']) 

# Делаем предсказание на загруженных данных (это нужно только для сигнатуры)
predictions = model.predict(X)
# Создаем сигнатуру и пример входных данных для MLflow
signature = infer_signature(X, predictions)
input_example = X[:10]
pip_requirements = "../requirements.txt"
metadata = {'model_type': 'baseline_regression'}
code_paths = ["base_model.ipynb"]

# Извлекаем параметры модели (используем правильное имя переменной - model)
model_params = model.named_steps["model"].get_params()

# Получаем id эксперимента или создаем новый, если его нет
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if not experiment:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:

    # Логируем гиперпараметры модели
    mlflow.log_params(model_params)

    # Логируем метрики, загруженные из cv_res.json
    mlflow.log_metrics(metrics)

    
    # Логируем саму модель и сразу регистрируем её в Model Registry
    # Так как модель использует sklearn Pipeline, используем mlflow.sklearn
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path='models',
        registered_model_name=REGISTRY_MODEL_NAME,
        signature=signature,
        input_example=input_example,
        pip_requirements=pip_requirements,
        metadata=metadata,
        code_paths=code_paths
    )

print(f"Запуск логирования завершен. Run ID: {run.info.run_id}")
print(f"Модель успешно зарегистрирована в реестре: {REGISTRY_MODEL_NAME}")

/home/mle-user/mle_projects/mle-project-sprint-2-v001/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Successfully registered model 'real_estate_model'.
2026/05/25 13:16:44 INFO mlflow.tracking._model_registry.client: Waiting up to 300 se

Запуск логирования завершен. Run ID: 05ee79bef2c84683b1a36b6cad05c022
Модель успешно зарегистрирована в реестре: real_estate_model


Created version '1' of model 'real_estate_model'.


In [8]:
print(f"EXPERIMENT_NAME: {EXPERIMENT_NAME}")
print(f"experiment_id: {experiment_id}")

EXPERIMENT_NAME: sprint2_project
experiment_id: 6
